# KNN Classifier
## Breast cancer wisconsin dataset
In this part, we will build a KNN classifier that takes an features as as input and outputs a label 0 or 1.

The breast cancer dataset contains 569 samples with 30 real, positive features (including cancer mass attributes like mean radius, mean texture, mean perimeter, et cetera). Of the samples, 212 are labeled “malignant” and 357 are labeled “benign”. 
You can find more details in: https://scikit-learn.org/stable/datasets/index.html#breast-cancer-dataset

In [59]:
from sklearn.datasets import load_breast_cancer
## Load the training set
data = load_breast_cancer()
X = data.data
y = data.target


In [60]:
y

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0,
       1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0,
       1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1,
       1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0,
       0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1,
       1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0,
       0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0,
       1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1,
       1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0,

In [ ]:
## print some statistics on the dataset
import numpy as np

print("Total number of samples: ", X.shape[0] ) #or y.shape[0]

# print the number of features per sample
print("Total number of featuers: ", X.shape[1])

# Total number of classes
print("Total number of classes: ", len(np.unique(y))) #get unique classes then count them

# print the number of samples in each class
unique, counts = np.unique(y, return_counts=True)

for label, count in zip(unique, counts):
    print("Class %s: %d samples" % (label, count))



Total number of samples:  569
Total number of featuers:  30
Total number of classes:  2
Class 0: 212 samples
Class 1: 357 samples


## Splitting the Train data to Train and Validate Sets

In [ ]:
from sklearn.model_selection import train_test_split

### 1. Split the data (i.e. features and target) into 70% train, 15% validate, and 15% test; Use Random Seed 777

# 70% train, 30% temp (validation + test)
trainx, tempx, trainy, tempy = train_test_split(X, y, test_size=0.30, stratify=y, random_state=777)

# Split temp evenly -> 15% validation + 15% test
valx, test_data, valy, test_labels = train_test_split(tempx, tempy, test_size=0.50, stratify=tempy, random_state=777)

print("Split sizes -> Train: %d | Validation: %d | Test: %d" % (trainx.shape[0], valx.shape[0], test_data.shape[0]))


Split sizes -> Train: 398 | Validation: 85 | Test: 86


## Nearest neighbor classification with L2 distance

To compute nearest neighbors in our data set, we need to first be able to compute distances between data points. A natural distance function is _Euclidean distance_: for two vectors $x, y \in \mathbb{R}^d$, their Euclidean distance is defined as 
$$\|x - y\| = \sqrt{\sum_{i=1}^d (x_i - y_i)^2}.$$
Often we omit the square root, and simply compute _squared Euclidean distance_:
$$\|x - y\|^2 = \sum_{i=1}^d (x_i - y_i)^2.$$
For the purposes of nearest neighbor computations, the two are equivalent: for three vectors $x, y, z \in \mathbb{R}^d$, we have $\|x - y\| \leq \|x - z\|$ if and only if $\|x - y\|^2 \leq \|x - z\|^2$.

## 1. Nearest neighbor classification with L2 distance

Here, we will write **NN_L2**, which takes as input the training data (`trainx` and `trainy`) and the test points (`evalx`) and predicts labels for these test points using 1-NN classification. These labels should be returned in a `numpy` array with one entry per test point. For **NN_L2**, the L2 norm should be used as the distance metric.

In [ ]:
def NN_L2(trainx, trainy, evalx):
    # inputs: trainx, trainy, testx <-- as defined above
    # output: an np.array of the predicted values for testy 
        
    ### 1. Computing pairwise Squared Euclidean (L2) distances between test and train samples
    squared_distances = ((evalx[:, None, :] - trainx[None, :, :]) ** 2).sum(axis=2)

    ### 2. Index of nearest training example
    nearest_index = squared_distances.argmin(axis=1)
    
    ### 3. Predicted label = label of nearest neighbor
    predictions = trainy[nearest_index]

    ### 4. Return predicted labels
    return predictions
    

## 2. K-Nearest neighbor classification with L2 distance

Here, we will write **KNN_L2**, which takes as input the training data (`trainx` and `trainy`), the test points (`evalx`), and the value of **K** (integer) and predicts labels for these test points using K-NN classification. These labels should be returned in a `numpy` array with one entry per test point.

In [64]:
def KNN_L2(trainx, trainy, evalx, K):
    # output: an np.array of the predicted values for testy 
    
    ### START CODE HERE ###

    
    ### 1. Computing pairwise Squared Euclidean (L2) distances between test and train samples    
    squared_distances = ((evalx[:, None, :] - trainx[None, :, :]) ** 2).sum(axis=2)

    ### 2. Index of nearest training example
    knn_indices = np.argpartition(squared_distances, K - 1, axis=1)[:, :K]
    
    ### 3. Getting labels
    neighbor_labels = trainy[knn_indices]

    ### 4. majority vote and return predictied labels
    predictions = (neighbor_labels.mean(axis=1) >= 0.5).astype(trainy.dtype)
    return predictions
    
    ### END CODE HERE ###

## 3. Nearest neighbor classification with L1 distance

We now compute nearest neighbors using the L1 distance (sometimes called *Manhattan Distance*).

Here, we will write a function, **NN_L1**, which again takes as input the arrays `trainx`, `trainy`, and `evalx`, and predicts labels for the test points using 1-nearest neighbor classification. For **NN_L1**, the L1 distance metric should be used. As before, the predicted labels should be returned in a `numpy` array with one entry per test point.

Notice that **NN_L1** and **NN_L2** may well produce different predictions on the test set.

In [ ]:
def NN_L1(trainx, trainy, evalx):
    # inputs: trainx, trainy, testx <-- as defined above
    # output: an np.array of the predicted values for testy 
        
    ### 1. Computing pairwise Manhattan (L1) distances between test and train samples
    manhattan_distances = np.abs(evalx[:, None, :] - trainx[None, :, :]).sum(axis=2)

    ### 2. Index of nearest training example 
    nearest_index = manhattan_distances.argmin(axis=1)

    ### 3. Getting labels 
    predictions = trainy[nearest_index]

    ### 4. Return predicted labels
    return predictions
    

## 4. K-Nearest neighbor classification with L1 distance

Here, we will write a function, **KNN_L1**, which takes as input the training data (`trainx` and `trainy`), the test points (`evalx`), and the value of **K** (integer) and predicts labels for these test points using K-NN classification and L1 distance metric. These labels should be returned in a `numpy` array with one entry per test point.

In [ ]:
def KNN_L1(trainx, trainy, evalx, K):
    # output: an np.array of the predicted values for testy 
    
    ### 1. Computing pairwise Manhattan (L1) distances between test and train samples
    manhattan_distances = np.abs(evalx[:, None, :] - trainx[None, :, :]).sum(axis=2)

    ### 2. Index of K nearest training example
    knn_indices  = np.argpartition(manhattan_distances, K - 1, axis=1)[:, :K]

    ### 3. Getting labels
    neighbor_labels = trainy[knn_indices]

    ### 4. majority vote and return predictied labels
    predictions = (neighbor_labels.mean(axis=1) >= 0.5).astype(trainy.dtype)
    return predictions
    
    

## 5. K-Nearest neighbor classifier

Here, we will write a function, **KNN**, which takes as input the training data (`trainx` and `trainy`), the test points (`evalx`), the value of **K** (integer), and a parameter for deciding the distance metric to be used (for example 1 for L1 and 2 for L2) and predicts labels for these test points using KNN classification. These labels should be returned in a `numpy` array with one entry per test point.

In [67]:
def KNN(trainx, trainy, evalx, K, dist_metric=2):
    # output: an np.array of the predicted values for testy 
    
    ### START CODE HERE ###

    if dist_metric == 1:
        distances = np.abs(evalx[:, None, :] - trainx[None, :, :]).sum(axis=2)
    elif dist_metric == 2:
        distances = ((evalx[:, None, :] - trainx[None, :, :]) ** 2).sum(axis=2)
    else:
        raise ValueError("Distance metric (dist_metric) must be either 1 (L1) or 2 (L2).")

    
    knn_indices = np.argpartition(distances, K - 1, axis=1)[:, :K]
        
    neighbor_labels = trainy[knn_indices]
                    
    predictions = (neighbor_labels.mean(axis=1) >= 0.5).astype(trainy.dtype)
    return predictions
    
    ### END CODE HERE ###

## 6. Putting it all together

Here we will write a code that allows you to select the hyper-parameters (distance measure and the value of K) by calling the KNN classifier with different values of K and either L1 or L2 distance measure. we need to make sure that you set the hyper-parameters using the validation set and not the test set. we need to systemtically try different values for K in conjunction with a distance measure and tabulate the results (you can do that be craeting a seperate cell and documenting in that cell) and note down the best hyper-parameter settings.

In [ ]:
import numpy as np
import time


k_values = [1,3,5,7,9,11,13,15,17]
distance_options = {1: "L1", 2: "L2"}

best_k = None
best_validation_accuracy = float("-inf")
best_distance_metric = None

for distance_metric in [1, 2]:
    for K in k_values:
        start = time.time()
        validation_predictions = KNN(trainx, trainy, valx, K, dist_metric = distance_metric)
        prediction_time = time.time() - start

        validation_accuracy = (validation_predictions == valy).mean()
        validation_error = 1.0 - validation_accuracy

        print("Distance Metric=%s | k = %-2d | Validation Accuracy: %.2f%% | Validation Error: %.2f%% | Evaluation CPU Time: %.4fs"
              % (distance_options[distance_metric], K, validation_accuracy * 100, validation_error * 100, prediction_time))

        if validation_accuracy > best_validation_accuracy:
            best_validation_accuracy = validation_accuracy
            best_k = K
            best_distance_metric = distance_metric

        ## For ties, we will prefer L2 since it is commonly used (got better accuracy after I did testing).
        elif np.isclose(validation_accuracy, best_validation_accuracy) and distance_metric == 2:
            best_k = K
            best_distance_metric = distance_metric

print("\nBest on validation -> Distance Metric = %s, k = %-2d, Accuracy = %.2f%%" % (distance_options[best_distance_metric], best_k, best_validation_accuracy * 100))


Distance Metric=L1 | k = 1  | Validation Accuracy: 94.12% | Validation Error: 5.88% | Evaluation CPU Time: 0.0097s
Distance Metric=L1 | k = 3  | Validation Accuracy: 96.47% | Validation Error: 3.53% | Evaluation CPU Time: 0.0062s
Distance Metric=L1 | k = 5  | Validation Accuracy: 96.47% | Validation Error: 3.53% | Evaluation CPU Time: 0.0056s
Distance Metric=L1 | k = 7  | Validation Accuracy: 96.47% | Validation Error: 3.53% | Evaluation CPU Time: 0.0053s
Distance Metric=L1 | k = 9  | Validation Accuracy: 96.47% | Validation Error: 3.53% | Evaluation CPU Time: 0.0032s
Distance Metric=L1 | k = 11 | Validation Accuracy: 95.29% | Validation Error: 4.71% | Evaluation CPU Time: 0.0039s
Distance Metric=L1 | k = 13 | Validation Accuracy: 95.29% | Validation Error: 4.71% | Evaluation CPU Time: 0.0026s
Distance Metric=L1 | k = 15 | Validation Accuracy: 95.29% | Validation Error: 4.71% | Evaluation CPU Time: 0.0024s
Distance Metric=L1 | k = 17 | Validation Accuracy: 94.12% | Validation Error: 5.

# 7. Test errors and the confusion matrix

Once the hyper-parameters have been selected, we now would like to perform a final evaluation on the test set and record the error rates. Also, we will write a function, **confusion**, which takes as input the true labels for the test set (that is, `testy`) as well as the predicted labels and returns the confusion matrix. The confusion matrix should be a `np.array`.

**Note:** we will record the cpu time it takes to perform the evaluation on the test set using functions like **time.time()**.

In [ ]:
def confusion(testy, testy_fit):
    # inputs: the correct labels, the fitted KNN labels 
    # output: a 10x10 np.array representing the confusion matrix as above
    
    ### 1. Determining the number of classes from largest label
    num_classes = int(max(testy.max(), testy_fit.max())) + 1
    
    ### 2. Creating an empty square matrix for counts
    matrix = np.zeros((num_classes, num_classes), dtype=int)
    

    ### 3. Completing the matrix counts
    for actual, predicted in zip(testy, testy_fit):
        matrix[int(actual), int(predicted)] += 1 # add one to the corresponding (true, predicted) cell

    return matrix
    

In [ ]:
# Code for performing the final evaluation on the test set and generating the confuson matrix.
import time


distance_name = "L1" if best_distance_metric == 1 else "L2"

# KNN "training": use all available labeled points (train + val)
X_final = np.vstack([trainx, valx])
y_final = np.concatenate([trainy, valy])

initial_time = time.time()
test_predictions = KNN(X_final, y_final, test_data, best_k, dist_metric=best_distance_metric)
prediction_time = time.time() - initial_time

test_accuracy = (test_predictions == test_labels).mean()
test_error = 1.0 - test_accuracy
confusion_matrix = confusion(test_labels, test_predictions)

print("Final Test -> Distance metric = %s | k = %-2d | Accuracy: %.2f%% | Error: %.2f%% | Prediction CPU Time: %.4fs"% (distance_name, best_k, test_accuracy * 100, test_error * 100, prediction_time))
print("\nConfusion matrix:\n", confusion_matrix)




Final Test -> Distance metric = L2 | k = 3  | Accuracy: 94.19% | Error: 5.81% | Prediction CPU Time: 0.0085s

Confusion matrix:
 [[29  3]
 [ 2 52]]


## 8. Analysis

1. What is the error rate on the validation set for NN_L2?
2. What is the best error rate on the validation set for KNN_L2?
3. What is the error rate on the validation set for NN_L1?
4. What is the best error rate on the validation set for KNN_L1?
5. What is the error rate on the test set?
7. Do you need to normalize data, in general, when using KNN?
8. Do you need to normalize data when using KNN for the given problem? Explain why?

In [72]:

### Q1: error on validation for NN_L2 
validation_prediction_nn_l2 = NN_L2(trainx, trainy, valx)
q1_error = 1.0 - (validation_prediction_nn_l2 == valy).mean()
print("Q1 Answer: Validation error (NN_L2) = %.2f%%" % (q1_error * 100))


### Q2: best error on validation for KNN_L2 (over k) 
k_values = [1,3,5,7,9,11,13,15,17]
best_error_knn_l2, best_k_l2 = float("inf"), None
for K in k_values:
    pred = KNN_L2(trainx, trainy, valx, K)
    err = 1.0 - (pred == valy).mean()
    if err < best_error_knn_l2:
        best_error_knn_l2, best_k_l2 = err, K
print("Q2 Answer: Best validation error (KNN_L2) = %.2f%% at k = %d" % (best_err_knn_l2 * 100, best_k_l2))

### Q3: error on validation for NN_L1 
val_pred_nn_l1 = NN_L1(trainx, trainy, valx)
q3_err = 1.0 - (val_pred_nn_l1 == valy).mean()
print("Q3 Answer: Validation error (NN_L1) = %.2f%%" % (q3_err * 100))

### Q4: best error on validation for KNN_L1 (over k)
best_err_knn_l1, best_k_l1 = float("inf"), None
for K in k_values:
    pred = KNN_L1(trainx, trainy, valx, K)
    err = 1.0 - (pred == valy).mean()
    if err < best_err_knn_l1:
        best_err_knn_l1, best_k_l1 = err, K
print("Q4 Answer: Best validation error (KNN_L1) = %.2f%% at k = %d" % (best_err_knn_l1 * 100, best_k_l1))

### Q5: error on the test set (use Section 6 choice; if tie there, L2 was preferred) 
# train final KNN on (train + val), then evaluate on test
try:
    final_k = best_k
    final_metric = best_distance_metric
except NameError:
    # fallback: pick between the two best validation errors found above (prefer L2 on tie)
    if np.isclose(best_err_knn_l1, best_err_knn_l2) or (best_err_knn_l2 < best_err_knn_l1):
        final_k, final_metric = best_k_l2, 2
    else:
        final_k, final_metric = best_k_l1, 1

X_final = np.vstack([trainx, valx])
y_final = np.concatenate([trainy, valy])

test_pred = KNN(X_final, y_final, test_data, final_k, dist_metric=final_metric)
q5_err = 1.0 - (test_pred == test_labels).mean()
print("Q5 Answer: Test error = %.2f%% (k = %d, metric = %s)" % (q5_err * 100, final_k, "L2" if final_metric == 2 else "L1"))

### Q6: normalize in general for KNN? 
print("Q6 Answer: Yes. KNN relies on distances, so unscaled features with large ranges dominate the distance.")

### Q7: normalize for this dataset? why? 
print("Q7 Answer: Yes. These breast-cancer features have different units/scales (e.g., radius, texture, area), " "so scaling puts them on the same footing and makes L1/L2 distances meaningful.")



Q1 Answer: Validation error (NN_L2) = 5.88%
Q2 Answer: Best validation error (KNN_L2) = 3.53% at k = 3
Q3 Answer: Validation error (NN_L1) = 5.88%
Q4 Answer: Best validation error (KNN_L1) = 3.53% at k = 3
Q5 Answer: Test error = 5.81% (k = 3, metric = L2)
Q6 Answer: Yes. KNN relies on distances, so unscaled features with large ranges dominate the distance.
Q7 Answer: Yes. These breast-cancer features have different units/scales (e.g., radius, texture, area), so scaling puts them on the same footing and makes L1/L2 distances meaningful.


## 9. Extra Stuff


1. Implementing weighted KNN where the vote of a neighbour is scaled down based on its distance from the test point.
2. Implement L_infinity distance measure and use it for classification.

## Points to remember

1. we will use numpy arrays and numpy libraries for efficient computations. 
2. Vectorize the code wherever possible instead of using explicit loops. This will significantly speed-up your code.

In [ ]:
import numpy as np, time

### Weighted KNN (inverse-distance vote), supports L1 and L2 

def KNN_weighted(trainx, trainy, evalx, K, dist_metric=2):
    # dist_metric: 1 -> L1 (Manhattan), 2 -> L2 (Euclidean; we use squared L2)
    if dist_metric == 1:
        distances = np.abs(evalx[:, None, :] - trainx[None, :, :]).sum(axis=2)
    elif dist_metric == 2:
        distances = ((evalx[:, None, :] - trainx[None, :, :]) ** 2).sum(axis=2)  # squared L2
    else:
        raise ValueError("dist_metric must be 1 (L1) or 2 (L2).")

    # K nearest indices for each eval point
    knn_idx = np.argpartition(distances, K - 1, axis=1)[:, :K]

    # gather neighbor labels and neighbor distances
    neighbor_labels = trainy[knn_idx]
    rows = np.arange(evalx.shape[0])[:, None]
    neighbor_dist  = distances[rows, knn_idx]

    # inverse-distance weights (avoid div-by-zero)
    weights = 1.0 / (neighbor_dist + 1e-12)

    # weighted probability of class 1
    prob1 = (weights * neighbor_labels).sum(axis=1) / weights.sum(axis=1)

    # final prediction (binary); ties (0.5) go to class 1 via >=
    predictions = (prob1 >= 0.5).astype(trainy.dtype)
    return predictions


#### 2. L_infinity (Chebyshev) distance: NN and KNN 

def NN_Linf(trainx, trainy, evalx):
    linf = np.max(np.abs(evalx[:, None, :] - trainx[None, :, :]), axis=2)
    nn_idx = np.argmin(linf, axis=1)
    return trainy[nn_idx]

def KNN_Linf(trainx, trainy, evalx, K):
    linf = np.max(np.abs(evalx[:, None, :] - trainx[None, :, :]), axis=2)
    knn_idx = np.argpartition(linf, K - 1, axis=1)[:, :K]
    neighbor_labels = trainy[knn_idx]
    preds = (neighbor_labels.mean(axis=1) >= 0.5).astype(trainy.dtype)
    return preds


#### A. Tune k for Weighted KNN (try L1 and L2; prefer L2 on ties)

k_values = [1,3,5,7,9,11,13,15,17]
best_k_w = None
best_dist_w = None
best_val_acc_w = float("-inf")
name = {1: "L1", 2: "L2"}

print("== Weighted KNN: validation sweep ==")
for dist_metric in [1, 2]:
    for K in k_values:
        t0 = time.time()
        val_pred = KNN_weighted(trainx, trainy, valx, K, dist_metric=dist_metric)
        pred_time = time.time() - t0
        val_acc = (val_pred == valy).mean()
        val_err = 1.0 - val_acc
        print("weighted | metric=%s | k = %-2d | Val Acc: %.2f%% | Val Err: %.2f%% | Pred CPU: %.4fs"
              % (name[dist_metric], K, val_acc * 100, val_err * 100, pred_time))

        # update if better; if tied, prefer L2
        if val_acc > best_val_acc_w:
            best_val_acc_w, best_k_w, best_dist_w = val_acc, K, dist_metric
        elif np.isclose(val_acc, best_val_acc_w) and dist_metric == 2:
            best_k_w, best_dist_w = K, dist_metric

print("\nBest (weighted) on validation -> metric=%s, k=%-2d, acc=%.2f%%\n"
      % (name[best_dist_w], best_k_w, best_val_acc_w * 100))

#### B. Evaluate Weighted KNN on the test set (time prediction only) ######

X_final = np.vstack([trainx, valx])
y_final = np.concatenate([trainy, valy])

t0 = time.time()
test_pred_w = KNN_weighted(X_final, y_final, test_data, best_k_w, dist_metric=best_dist_w)
test_cpu_w = time.time() - t0
test_acc_w = (test_pred_w == test_labels).mean()
test_err_w = 1.0 - test_acc_w

print("Weighted KNN -> metric=%s | k=%-2d | Test Acc: %.2f%% | Test Err: %.2f%% | Pred CPU: %.4fs"
      % (name[best_dist_w], best_k_w, test_acc_w * 100, test_err_w * 100, test_cpu_w))


#### C. Tune k for KNN with L_infinity distance ######

best_k_inf = None
best_val_acc_inf = float("-inf")

print("\n== KNN with L_infinity: validation sweep ==")
for K in k_values:
    t0 = time.time()
    val_pred = KNN_Linf(trainx, trainy, valx, K)
    pred_time = time.time() - t0
    val_acc = (val_pred == valy).mean()
    val_err = 1.0 - val_acc
    print("L_inf   | k = %-2d | Val Acc: %.2f%% | Val Err: %.2f%% | Pred CPU: %.4fs"
          % (K, val_acc * 100, val_err * 100, pred_time))
    if val_acc > best_val_acc_inf:
        best_val_acc_inf, best_k_inf = val_acc, K

print("\nBest (L_inf) on validation -> k=%-2d, acc=%.2f%%\n" % (best_k_inf, best_val_acc_inf * 100))

#### D. Evaluate L_infinity KNN on the test set ######

t0 = time.time()
test_pred_inf = KNN_Linf(X_final, y_final, test_data, best_k_inf)
test_cpu_inf = time.time() - t0
test_acc_inf = (test_pred_inf == test_labels).mean()
test_err_inf = 1.0 - test_acc_inf

print("KNN (L_inf) -> k=%-2d | Test Acc: %.2f%% | Test Err: %.2f%% | Pred CPU: %.4fs"
      % (best_k_inf, test_acc_inf * 100, test_err_inf * 100, test_cpu_inf))


== Weighted KNN: validation sweep ==
weighted | metric=L1 | k = 1  | Val Acc: 94.12% | Val Err: 5.88% | Pred CPU: 0.0086s
weighted | metric=L1 | k = 3  | Val Acc: 94.12% | Val Err: 5.88% | Pred CPU: 0.0062s
weighted | metric=L1 | k = 5  | Val Acc: 94.12% | Val Err: 5.88% | Pred CPU: 0.0062s
weighted | metric=L1 | k = 7  | Val Acc: 96.47% | Val Err: 3.53% | Pred CPU: 0.0047s
weighted | metric=L1 | k = 9  | Val Acc: 96.47% | Val Err: 3.53% | Pred CPU: 0.0042s
weighted | metric=L1 | k = 11 | Val Acc: 95.29% | Val Err: 4.71% | Pred CPU: 0.0045s
weighted | metric=L1 | k = 13 | Val Acc: 95.29% | Val Err: 4.71% | Pred CPU: 0.0041s
weighted | metric=L1 | k = 15 | Val Acc: 95.29% | Val Err: 4.71% | Pred CPU: 0.0040s
weighted | metric=L1 | k = 17 | Val Acc: 94.12% | Val Err: 5.88% | Pred CPU: 0.0031s
weighted | metric=L2 | k = 1  | Val Acc: 94.12% | Val Err: 5.88% | Pred CPU: 0.0022s
weighted | metric=L2 | k = 3  | Val Acc: 92.94% | Val Err: 7.06% | Pred CPU: 0.0033s
weighted | metric=L2 | k = 5